# Meeting minutes

The lesson takes a meeting recording, transcribes it, then writes the notes: summary, discussion, decisions, and action items.

The course Colab uses Whisper for the audio and Llama for the notes. Here Whisper stays the audio step, through the Azure deployment `WHISPER_MODEL`. The notes come from an Azure chat deployment instead of Llama.

[Course Colab](https://colab.research.google.com/drive/1KSMxOCprsl1QRpt_Rq0UqCAyMtPqDQYx?usp=sharing)


## Client

In [1]:
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)
whisper_model = os.getenv("WHISPER_MODEL")
deployments = ["gpt-4o-mini", "gpt-4o", "gpt-4.1"]


## Transcribe, then write the notes

In [2]:
def transcribe(audio_path):
    if not audio_path or not whisper_model:
        return ""
    with open(audio_path, "rb") as audio_file:
        result = client.audio.transcriptions.create(
            model=whisper_model,
            file=audio_file,
            timeout=120.0,
        )
    return result.text or ""


def write_minutes(transcript, deployment):
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {
                "role": "system",
                "content": (
                    "Write meeting minutes in markdown. "
                    "Include a short summary, discussion points, decisions, "
                    "and action items with owners when the transcript gives them."
                ),
            },
            {"role": "user", "content": transcript},
        ],
        timeout=120.0,
    )
    return response.choices[0].message.content


def minutes_from(audio_path, pasted_text, deployment):
    transcript = (pasted_text or "").strip() or transcribe(audio_path)
    if not transcript:
        return "Add an audio file, or paste a transcript. Audio needs WHISPER_MODEL in .env."
    return write_minutes(transcript, deployment)


## Page

In [3]:
audio = gr.Audio(label="Recording", type="filepath")
pasted = gr.Textbox(label="Or paste a transcript", lines=8)
deployment = gr.Dropdown(deployments, value="gpt-4o-mini", label="Notes model")
notes = gr.Markdown(label="Minutes")

gr.Interface(
    fn=minutes_from,
    inputs=[audio, pasted, deployment],
    outputs=notes,
    title="Meeting minutes",
    flagging_mode="never",
).launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
